# 04 · Code evaluation on a GPU

Runs the executable code evaluation (`evals/code_tasks.jsonl`, 40 tasks with unit tests) for every model and prompt mode in one session, with identical settings, and uploads the receipts to the runs repo under `evals/`.

On a laptop CPU one evaluation takes 30-120 minutes; here all six take about half an hour.

**Runtime:** *Select Kernel → Colab → New Colab Server → **L4*** (an A100 is not needed; L4 costs fewer units).

Afterwards, locally: `python scripts/fetch_evals.py` copies the receipts into `artifacts/evals/`.

## 1. GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Free the GPU

Must run before anything touches CUDA in this kernel.

In [ ]:
import os
import signal

me = os.getpid()
ps = subprocess.run(["ps", "-eo", "pid,args"], capture_output=True, text=True).stdout.splitlines()[1:]
stale = []
for line in ps:
    pid_text, _, args = line.strip().partition(" ")
    pid = int(pid_text)
    if pid != me and ("colab_kernel_launcher" in args or ("compile_worker" in args and f"--parent={me}" not in args)):
        stale.append(pid)
for pid in stale:
    os.kill(pid, signal.SIGKILL)
print("stale processes killed:", len(stale))

## 3. Get the code

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
for path in (REPO_DIR, REPO_DIR / "scripts"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

## 4. Test the evaluation code

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_code_eval.py"], capture_output=True, text=True)
print(result.stdout[-1500:], result.stderr[-500:])
assert result.returncode == 0, "tests failed"

## 5. Log in to Hugging Face

Write access to `glm-ladder-runs` is needed to upload the receipts.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

RUNS_REPO = "Tbt-AI/glm-ladder-runs"
api = HfApi(token=getpass("Hugging Face write token: "))
print("logged in as:", api.whoami()["name"])
assert api.repo_info(RUNS_REPO, repo_type="model").private

## 6. Evaluate

Every (run, mode) pair with the same settings: reference-solution loss, 1 greedy answer and 10 samples at temperature 0.8 per task. Each run's checkpoint is downloaded once and checked against its SHA-256.

In [ ]:
import json
import time

import torch

from glm_ladder.code_eval import load_tasks
from glm_ladder.tokenizer import load_tokenizer
from run_code_eval import evaluate, receipt
from sample import load_run

EVALS = [
    ("s50m-001", "completion"), ("sft-s50m-001", "completion"), ("sft-s50m-001", "chat"),
    ("m100m-001", "completion"), ("sft-m100m-001", "completion"), ("sft-m100m-001", "chat"),
]
SETTINGS = dict(n=10, temperature=0.8, top_k=50, max_tokens=160, seed=0, loss_only=False)

torch.set_float32_matmul_precision("high")
device = torch.device("cuda")
gpu = torch.cuda.get_device_name()
tokenizer = load_tokenizer(REPO_DIR / "tokenizer" / "glm-ladder-16k.json")
tasks = load_tasks(REPO_DIR / "evals" / "code_tasks.jsonl")
OUT = Path("/content/evals")
OUT.mkdir(exist_ok=True)

summaries = {}
loaded = {}
for run, mode in EVALS:
    if run not in loaded:
        loaded.clear()
        torch.cuda.empty_cache()
        model, state = load_run(api, RUNS_REPO, run, None)
        loaded[run] = (model.to(device), state)
    model, state = loaded[run]
    started = time.perf_counter()
    results, summary = evaluate(model, tokenizer, tasks, mode=mode, log=lambda _: None, **SETTINGS)
    data = receipt(run, state["checkpoint"], mode, results, summary, device=gpu, **SETTINGS)
    data["commit"] = commit
    (OUT / f"code-{run}-{mode}.json").write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")
    summaries[(run, mode)] = summary
    print(f"{run:14s} {mode:10s} {time.perf_counter() - started:5.0f} s  "
          + "  ".join(f"{key} {value:.3f}" for key, value in summary.items()))

## 7. Table and upload

In [ ]:
keys = ["reference_loss", "greedy_pass@1", "sampled_pass@1", "sampled_pass@5", "sampled_pass@10"]
print("| run | mode | " + " | ".join(keys) + " |")
print("|---|---|" + "---:|" * len(keys))
for (run, mode), summary in summaries.items():
    print(f"| {run} | {mode} | " + " | ".join(f"{summary[key]:.3f}" for key in keys) + " |")

api.upload_folder(folder_path=str(OUT), path_in_repo="evals", repo_id=RUNS_REPO, repo_type="model",
                  commit_message=f"code evals on {gpu}, commit {commit}")
print("uploaded to", RUNS_REPO, "evals/")

## 8. Done

Close the runtime. Then, locally:

```bash
python scripts/fetch_evals.py
git add artifacts/evals/
```